<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/06_observation_scheduler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AstroPlanner AI: Observation Scheduler
## Phase 5 — Schedules observations for user

In [36]:
# Cell: schedule-setup
!pip install requests -q

import sys, os, json
from datetime import datetime, timedelta
from google.colab import drive

drive.mount('/content/drive')

!git clone https://github.com/Arfa-Tariq/AstroPlanner-AI.git 2>/dev/null || git -C /content/AstroPlanner-AI pull

sys.path.append('/content/AstroPlanner-AI/src')

DATA_DIR = '/content/drive/MyDrive/AstroPlanner/data'
os.makedirs(DATA_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.


In [37]:
# Cell: schedule-imports
from models import WeeklyPlanRequest, UserProfile

In [38]:
# Cell: schedule-load
with open(f'{DATA_DIR}/current_request.json') as f:
    plan_request = WeeklyPlanRequest.model_validate_json(f.read())

with open(f'{DATA_DIR}/weekly_fov_analysis.json') as f:
    fov_output = json.load(f)

weekly_nights = fov_output['nights']
setup_summary = fov_output['setup_summary']

print(f"User          : {plan_request.user.name}")
print(f"Nights loaded : {len(weekly_nights)}")
print(f"Camera on file: {'yes' if setup_summary else 'no'}")

User          : Andrew
Nights loaded : 7
Camera on file: yes


In [39]:
# Cell: schedule-time-parsing
def parse_local_time(time_str: str, night_date: str) -> "datetime | None":
    """
    Converts the mixed time formats produced upstream into real datetimes
    on a shared timeline so objects can be sorted and checked for overlap.

    Handles three shapes seen across notebooks 03/04:
    - 'HH:MM'          (deep-sky peak_time_local / above_30deg_from_local)
    - 'MM/DD HH:MM'     (planet transit_time / rise_time / set_time)
    - non-time strings  ('already above 30° at dusk', 'up at dawn',
                          'still up at window end', 'not visible this window',
                          'circumpolar', etc.) -> returns None, caller decides
                          how to handle missing bounds.

    A night can span two calendar dates (e.g. 22:00 on the 27th through
    05:00 on the 28th) — HH:MM-only strings before 12:00 are assumed to be
    the following morning, matching how notebook 03 constructs its windows.
    """
    if not time_str or not isinstance(time_str, str):
        return None

    year, month, day = int(night_date[:4]), int(night_date[5:7]), int(night_date[8:10])

    if "/" in time_str:  # 'MM/DD HH:MM'
        try:
            md, hm = time_str.split(" ")
            mm, dd = md.split("/")
            hh, mi = hm.split(":")
            return datetime(year, int(mm), int(dd), int(hh), int(mi))
        except Exception:
            return None

    if ":" in time_str and len(time_str) <= 5:  # 'HH:MM'
        try:
            hh, mi = time_str.split(":")
            hh, mi = int(hh), int(mi)
            dt = datetime(year, month, day, hh, mi)
            if hh < 12:  # early-morning hours belong to the next calendar day
                dt += timedelta(days=1)
            return dt
        except Exception:
            return None

    return None  # non-parseable label like 'up at dawn' — handled by caller

In [40]:
# Cell: schedule-window
DEFAULT_SESSION_MINUTES = {
    "planetary_target": 25,   # video capture/stacking session
    "fits_well": 40,
    "too_small": 30,
    "too_large": 45,          # mosaics/wide framing take longer
    "unknown": 30,
    "no_camera": 20,          # visual-only, quick eyepiece look
}


def get_object_window(obj: dict, night_date: str) -> tuple:
    """
    Returns (start_dt, end_dt, peak_dt) for one object's observing window,
    reconciling the different field names used for solar-system bodies
    (rise_time/set_time/transit_time) vs. deep-sky objects
    (rise_time_local/set_time_local or above_30deg_from_local/until_local,
    peak_time_local). Falls back to a window centered on peak time when a
    boundary is a non-parseable label ('up at dusk', 'circumpolar', etc.)
    rather than dropping the object.
    """
    is_solar = obj.get('is_solar_system', False)

    if is_solar:
        start = parse_local_time(obj.get('rise_time'), night_date)
        end = parse_local_time(obj.get('set_time'), night_date)
        peak = parse_local_time(obj.get('transit_time'), night_date)
    else:
        start = parse_local_time(
            obj.get('above_30deg_from_local') or obj.get('rise_time_local'), night_date
        )
        end = parse_local_time(
            obj.get('above_30deg_until_local') or obj.get('set_time_local'), night_date
        )
        peak = parse_local_time(obj.get('peak_time_local'), night_date)

    if peak is None:
        return None, None, None  # can't schedule without at least a peak time

    if start is None:
        start = peak - timedelta(minutes=30)
    if end is None:
        end = peak + timedelta(minutes=30)

    return start, end, peak

In [41]:
def build_night_schedule(night: dict, max_objects: int = 8) -> dict:
    date_str = night['date']
    night_scheduled, day_bonus = [], []

    # True dark-window bounds for this night, derived from ALL deep-sky
    # objects (not just the first one) — a single object's own window can
    # be narrower than the full night (e.g. rises late, sets early), which
    # previously caused legitimately-dark peak times near dawn to be
    # misclassified as daytime.
    dark_start, dark_end = None, None
    for obj in night['recommended_objects']:
        if obj.get('is_solar_system'):
            continue
        s, e, _ = get_object_window(obj, date_str)
        if s is None or e is None:
            continue
        dark_start = s if dark_start is None else min(dark_start, s)
        dark_end = e if dark_end is None else max(dark_end, e)

    for obj in night['recommended_objects']:
        start, end, peak = get_object_window(obj, date_str)
        if peak is None:
            continue

        period = obj.get('observable_period', 'night')
        fov_fit = obj.get('fov_analysis', {}).get('fov_fit', 'unknown')
        duration = timedelta(minutes=DEFAULT_SESSION_MINUTES.get(fov_fit, 30))

        slot_start = max(start, peak - duration / 2)
        slot_end = min(end, slot_start + duration)
        if slot_end <= slot_start:
            continue

        peak_is_dark = (
            dark_start is not None and dark_end is not None
            and dark_start <= peak <= dark_end
        )

        entry = {
            'name': obj['name'], 'common_name': obj.get('common_name'),
            'target_type': obj.get('target_type'),
            'recommendation_score': obj.get('recommendation_score'),
            'fov_fit': fov_fit, 'observable_period': period,
            'start': slot_start, 'end': slot_end, 'peak': peak,
            'note': obj.get('fov_analysis', {}).get('note', ''),
        }

        if obj.get('is_solar_system') and not peak_is_dark:
            day_bonus.append(entry)
            continue

        overlaps = any(slot_start < s['end'] and slot_end > s['start'] for s in night_scheduled)
        if overlaps:
            continue
        night_scheduled.append(entry)
        if len(night_scheduled) >= max_objects:
            break

    night_scheduled.sort(key=lambda s: s['start'])
    day_bonus.sort(key=lambda s: s['start'])
    return {'night_session': night_scheduled, 'daytime_bonus': day_bonus[:3]}

In [42]:
def get_weekly_schedule(weekly_nights: list, max_objects: int = 8) -> list[dict]:
    def fmt(slot):
        return {**{k: v for k, v in slot.items() if k not in ('start', 'end', 'peak')},
                'start_local': slot['start'].strftime('%H:%M'),
                'end_local': slot['end'].strftime('%H:%M'),
                'peak_local': slot['peak'].strftime('%H:%M')}

    weekly = []
    for night in weekly_nights:
        result = build_night_schedule(night, max_objects=max_objects)
        weekly.append({
            'date': night['date'],
            'timeline': [fmt(s) for s in result['night_session']],
            'daytime_bonus': [fmt(s) for s in result['daytime_bonus']],
        })
    return weekly

weekly_schedule = get_weekly_schedule(weekly_nights)

with open(f'{DATA_DIR}/weekly_observation_schedule.json', 'w') as f:
    json.dump(weekly_schedule, f, indent=2, default=str)

print(f"Saved to {DATA_DIR}/weekly_observation_schedule.json\n")

Saved to /content/drive/MyDrive/AstroPlanner/data/weekly_observation_schedule.json



In [43]:
best = max(weekly_schedule, key=lambda n: len(n['timeline']))
print(f"Best night: {best['date']}  ({len(best['timeline'])} scheduled sessions)\n")

print("=== Night Session (astronomical darkness) ===")
for slot in best['timeline']:
    label = slot['common_name'] or slot['name']
    print(f"  {slot['start_local']}–{slot['end_local']}  {slot['name']:10} {label:20} score={slot['recommendation_score']}")
    if slot['note']:
        print(f"      {slot['note']}")

if best['daytime_bonus']:
    print("\n=== Daytime Bonus Targets (optional, advanced) ===")
    for slot in best['daytime_bonus']:
        print(f"  {slot['start_local']}–{slot['end_local']}  {slot['name']} — visible in daylight, needs a solar-safe finder/filter awareness for setup")

Best night: 2026-07-30  (5 scheduled sessions)

=== Night Session (astronomical darkness) ===
  23:15–23:45  NGC6813    NGC6813              score=0.3971
      This object is only 0.05° across — about 0.5% of your frame width. It'll appear as a small feature in the frame; consider a longer focal length or a Barlow/reducer.
  00:23–00:53  NGC6974    NGC6974              score=0.3867
      No cataloged size for this object — fit could not be assessed.
  00:56–01:26  NGC6995    Eastern Veil,Network Nebula score=0.3088
      This object is only 0.20° across — about 2.1% of your frame width. It'll appear as a small feature in the frame; consider a longer focal length or a Barlow/reducer.
  01:27–01:52  Moon       Moon                 score=0.827
      Solar system target — use high-frame-rate planetary/lunar capture and stacking rather than single-frame deep-sky FoV framing. Current pixel scale is undersampled for this technique (11.09"/px).
  02:04–02:34  NGC7293    Helix Nebula         sc